# Chapter 5 lab: Alpha and conditional expectation
**Reader prototype 0.1 · companion to the v5.7 discussion · about 15 minutes**

Run the cells from top to bottom. Change the **Parameters** cell and run again.
This exact, finite probability model uses **synthetic returns**; it does not estimate an investable strategy or request market data.

You will see why a conditional alpha forecast is a random variable, why the residual has conditional mean zero, and how information changes prediction error.

**Quick experiment:** run the defaults, then change `BENCHMARK_PP` from `3.0` to `1.0`.
Does the alpha change? Does the unpredictable residual change?

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from QuantConnect.Research import QuantBook
except ImportError:
    print("Runtime: local Python (QuantConnect is not installed here).")
else:
    qb = QuantBook()
    print("Runtime: QuantConnect QuantBook initialized; no market data needed.")
print("Python:", sys.version.split()[0])
plt.rcParams.update({"font.size": 11, "figure.dpi": 130,
                     "axes.spines.top": False, "axes.spines.right": False})

## 1. Probability space and maps
Let $\Omega=\{\omega_1,\ldots,\omega_4\}$, $\mathcal F=2^\Omega$, and $\mathbb P(\{\omega_i\})=p_i$.
The full **probability space** is $(\Omega,\mathcal F,\mathbb P)$.

$$X:(\Omega,\mathcal F)\longrightarrow(\mathbb R,\mathcal B(\mathbb R)),\quad \omega_i\longmapsto x_i.$$

$X$ is tomorrow's return. The group $Y$ is observable today; its information is $\mathcal G=\sigma(Y)\subseteq\mathcal F$.
With a constant benchmark forecast $b$, define

$$M=\mathbb E[X\mid\mathcal G],\quad \alpha=M-b,\quad \varepsilon=X-M.$$
$$\alpha:(\Omega,\mathcal G)\longrightarrow(\mathbb R,\mathcal B(\mathbb R)),\quad \omega_i\longmapsto M(\omega_i)-b.$$

Thus $X=b+\alpha+\varepsilon$. Before observing today's group, $\alpha$ is a random variable; after observing it, we have a particular forecast value.

More generally, an integrable, $\mathcal G$-measurable $b(Y)$ can replace the constant benchmark.
This is a **benchmark-dependent conditional alpha forecast**, distinct from the constant Jensen alpha in a factor regression.

### Parameters — edit this next cell
All returns are in **percentage points**: `3.0` means a 3% return. Probabilities must be positive and sum to one.
`INFORMATION` is `"group"`, `"none"`, or `"full"`. **Full outcome information is an oracle thought experiment, unavailable before tomorrow's return.**

In [ ]:
RETURN_PP = [0.0, 2.0, 4.0, 6.0]
PROBABILITIES = [0.25, 0.25, 0.25, 0.25]
GROUPS = ["A", "A", "B", "B"]
BENCHMARK_PP = 3.0
INFORMATION = "group"

## 2. Compute the conditional expectation exactly
For each group $A$ of positive probability,
$$M(\omega_i)=\frac{\sum_{j\in A}p_jx_j}{\sum_{j\in A}p_j},\qquad i\in A.$$
These are weighted group means, not fitted sample means. Changing the probabilities changes the specified population model.

In [ ]:
def conditional_mean(x, p, labels):
    x, p, labels = np.asarray(x, float), np.asarray(p, float), np.asarray(labels)
    if x.ndim != 1 or p.shape != x.shape or labels.shape != x.shape or not len(x):
        raise ValueError("Returns, probabilities, and labels need the same nonzero length.")
    if not np.isfinite(x).all() or not np.isfinite(p).all():
        raise ValueError("Use finite returns and probabilities.")
    if np.any(p <= 0) or not np.isclose(p.sum(), 1, rtol=0, atol=1e-12):
        raise ValueError("Probabilities must be positive and sum to one.")
    result = np.empty_like(x)
    for label in np.unique(labels):
        mask = labels == label
        result[mask] = np.dot(p[mask], x[mask]) / p[mask].sum()
    return result

x = np.array(RETURN_PP, dtype=float)
p = np.array(PROBABILITIES, dtype=float)
groups = np.array(GROUPS, dtype=str)
labels_by_mode = {"none": np.repeat("All", len(x)), "group": groups,
                  "full": np.array([str(i) for i in range(len(x))])}
if INFORMATION not in labels_by_mode:
    raise ValueError("INFORMATION must be none, group, or full.")
if not np.isfinite(BENCHMARK_PP):
    raise ValueError("The benchmark must be finite.")
# Validate the original groups even when another information mode is selected.
conditional_mean(x, p, groups)
labels = labels_by_mode[INFORMATION]
m = conditional_mean(x, p, labels)
alpha = m - BENCHMARK_PP
residual = x - m
table = pd.DataFrame({"Outcome": [f"w{i+1}" for i in range(len(x))],
    "Probability": p, "Information group": labels, "Return (%)": x,
    "Forecast (%)": m, "Benchmark (%)": BENCHMARK_PP,
    "Alpha (pp)": alpha, "Residual (pp)": residual})
display(table.round(4))
print("Information:", INFORMATION)
if INFORMATION == "full":
    print("ORACLE: tomorrow's outcome is revealed. This is not a usable trading signal.")

## 3. See forecast, alpha, and error
The residual need not vanish at each outcome. Its **conditional average within each information group** vanishes.
Negative alpha means below-benchmark expected return, not necessarily a negative expected return.

In [ ]:
means = {name: conditional_mean(x, p, lab) for name, lab in labels_by_mode.items()}
errors = {name: float(np.dot(p, (x - pred)**2)) for name, pred in means.items()}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
idx = np.arange(len(x))
axes[0].bar(idx - .18, x, .36, label="Return X", color="#90AFC5")
axes[0].bar(idx + .18, m, .36, label="Forecast M", color="#235789")
axes[0].axhline(BENCHMARK_PP, color="#BD6635", ls="--", label="Benchmark")
axes[0].set(title="Return and conditional forecast", ylabel="Return (%)")
axes[0].legend(fontsize=8)
axes[1].bar(idx - .18, alpha, .36, label="Alpha M - b", color="#27877B")
axes[1].bar(idx + .18, residual, .36, label="Residual X - M", color="#C276A0")
axes[1].set(title="Predictable difference and residual", ylabel="Percentage points")
axes[1].legend(fontsize=8)
for ax in axes[:2]:
    ax.axhline(0, color="#555555", lw=.7)
    ax.set_xticks(idx, table["Outcome"])
    ax.set_xlabel("Possible outcome")
names = ["None", "Group", "Full outcome\n(oracle only)"]
bars = axes[2].bar(names, list(errors.values()), color=["#90AFC5", "#235789", "#A9AAA9"])
axes[2].bar_label(bars, fmt="%.3g", padding=4)
axes[2].set(title="More information, less error", ylabel="Mean squared error (pp squared)",
            ylim=(0, max(1, max(errors.values()) * 1.25)))
fig.suptitle(f"Conditional alpha: {INFORMATION} information | synthetic probability model", fontsize=14)
plt.show()
display(pd.DataFrame({"Information": list(errors), "MSE (pp squared)": list(errors.values())}))

## 4. Why $L^2$ appears
The conditional-expectation operator can be written
$$\mathbb E[\,\cdot\mid\mathcal G]:L^2(\Omega,\mathcal F,\mathbb P)\longrightarrow L^2(\Omega,\mathcal G,\mathbb P),\quad X\longmapsto M.$$
Elements of $L^2$ are equivalence classes of square-integrable random variables, identified up to equality almost surely. We compute with representatives.

$M$ is the orthogonal projection of $X$ onto the subspace of $\mathcal G$-measurable square-integrable random variables. For a square-integrable benchmark, $\alpha=M-b$ is also in that subspace. Square integrability is an **assumption**, not part of the meaning of alpha. Here it holds because there are finitely many finite returns.

$$\mathbb E[\varepsilon\mid\mathcal G]=0,\quad \mathbb E[\varepsilon Z]=0\quad (Z\in L^2(\mathcal G)).$$
$$\mathbb E[(X-b)^2]=\mathbb E[\alpha^2]+\mathbb E[\varepsilon^2].$$
The next cell checks conditional means and the error decomposition numerically. Checking each group indicator establishes orthogonality to **every function of the group** in this finite model.

In [ ]:
conditional_residual = conditional_mean(residual, p, labels)
lhs = np.dot(p, (x - BENCHMARK_PP)**2)
predictable = np.dot(p, alpha**2)
unpredictable = np.dot(p, residual**2)
assert np.allclose(x, BENCHMARK_PP + alpha + residual)
assert np.allclose(conditional_residual, 0, atol=1e-12)
assert np.isclose(np.dot(p, m), np.dot(p, x))
assert np.isclose(lhs, predictable + unpredictable)
assert errors["none"] + 1e-10 >= errors["group"] >= errors["full"] - 1e-10
print("PASS: decomposition, conditional residual, tower property, orthogonality, nested MSE.")
print(f"Squared error about benchmark: {lhs:.4g} = {predictable:.4g} + {unpredictable:.4g} pp squared")

## 5. Three experiments
1. **Benchmark:** change `BENCHMARK_PP` to `1.0`. Explain why alpha changes but $M$, the residual, and prediction MSE do not.
2. **Unequal probabilities:** try `[0.1, 0.4, 0.2, 0.3]`. Compute both weighted group means by hand before rerunning.
3. **Information:** compare `none`, `group`, and `full`. Why does zero error with `full` not demonstrate a tradable edge?

### Check your answers
With the defaults, $M=(1,1,5,5)$, $\alpha=(-2,-2,2,2)$, and $\varepsilon=(-1,1,-1,1)$, all in percentage points. The three MSE values are $5,1,0$ in squared percentage points.
With benchmark 1, alpha becomes $(0,0,4,4)$; the residual is unchanged.
With the proposed unequal probabilities, the group means become $1.6$ and $5.2$.

### From this exact model to research
In data, we estimate the unknown conditional expectation using past information. A fitted forecast does not automatically inherit the exact zero-conditional-residual property. Out-of-sample evaluation, risk, costs, and estimation uncertainty remain necessary. No profit or Sharpe claim follows from this toy example.

**References:** David Williams, *Probability with Martingales*, Chapter 9 (conditional expectation); [QuantConnect Alpha Model concepts](https://www.quantconnect.com/docs/v2/writing-algorithms/algorithm-framework/alpha/key-concepts).
QuantConnect uses “Alpha Model” more broadly for a model emitting predictive Insights; it does not require the specific mathematical definition used here.